[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/Optimization-for-Machine-Learning/blob/main/notebooks/ch04_advanced_optimizers.ipynb)

# Chapter 4: Advanced Optimizers

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Implement from scratch**: Momentum, RMSprop, AdaGrad, and Adam optimizers
2. **Understand** the mathematical foundations behind each optimizer
3. **Visualize** how different optimizers navigate loss surfaces
4. **Compare** optimizer performance on various objective functions
5. **Select** the appropriate optimizer for different machine learning tasks
6. **Tune** hyperparameters effectively for each optimizer

---

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import LogNorm
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# For reproducibility
np.random.seed(42)

print("Setup complete!")

## 1. The Problem with Vanilla Gradient Descent

Before diving into advanced optimizers, let's understand why we need them. Vanilla gradient descent has several limitations:

- **Slow convergence** in ravines (elongated valleys)
- **Oscillations** when gradients vary significantly across dimensions
- **Fixed learning rate** doesn't adapt to the loss landscape
- **Gets stuck** in saddle points and local minima

Let's visualize these problems:

In [ ]:
# Define test functions for optimization

def rosenbrock(x, y, a=1, b=100):
    """Rosenbrock function - a classic test for optimization algorithms.
    Global minimum at (a, a^2) = (1, 1) with value 0.
    """
    return (a - x)**2 + b * (y - x**2)**2

def rosenbrock_grad(x, y, a=1, b=100):
    """Gradient of Rosenbrock function."""
    dx = -2*(a - x) - 4*b*x*(y - x**2)
    dy = 2*b*(y - x**2)
    return np.array([dx, dy])

def beale(x, y):
    """Beale function - has a flat region that can trap optimizers.
    Global minimum at (3, 0.5) with value 0.
    """
    return ((1.5 - x + x*y)**2 + 
            (2.25 - x + x*y**2)**2 + 
            (2.625 - x + x*y**3)**2)

def beale_grad(x, y):
    """Gradient of Beale function (numerical approximation)."""
    eps = 1e-7
    dx = (beale(x + eps, y) - beale(x - eps, y)) / (2 * eps)
    dy = (beale(x, y + eps) - beale(x, y - eps)) / (2 * eps)
    return np.array([dx, dy])

def saddle(x, y):
    """Saddle point function - tests ability to escape saddle points.
    Saddle point at (0, 0).
    """
    return x**2 - y**2

def saddle_grad(x, y):
    """Gradient of saddle function."""
    return np.array([2*x, -2*y])

def quadratic_bowl(x, y, condition_number=10):
    """Ill-conditioned quadratic - tests handling of different curvatures.
    Global minimum at (0, 0) with value 0.
    """
    return x**2 + condition_number * y**2

def quadratic_bowl_grad(x, y, condition_number=10):
    """Gradient of quadratic bowl."""
    return np.array([2*x, 2*condition_number*y])

print("Test functions defined!")

In [ ]:
# Visualize the test functions

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Rosenbrock
x = np.linspace(-2, 2, 200)
y = np.linspace(-1, 3, 200)
X, Y = np.meshgrid(x, y)
Z = rosenbrock(X, Y)
axes[0, 0].contour(X, Y, Z, levels=np.logspace(-1, 3, 30), norm=LogNorm(), cmap='viridis')
axes[0, 0].plot(1, 1, 'r*', markersize=15, label='Global minimum')
axes[0, 0].set_title('Rosenbrock Function\n(Narrow curved valley)', fontsize=14)
axes[0, 0].set_xlabel('x')
axes[0, 0].set_ylabel('y')
axes[0, 0].legend()

# Beale
x = np.linspace(-4.5, 4.5, 200)
y = np.linspace(-4.5, 4.5, 200)
X, Y = np.meshgrid(x, y)
Z = beale(X, Y)
Z = np.clip(Z, 0, 1000)  # Clip for better visualization
axes[0, 1].contour(X, Y, Z, levels=np.logspace(0, 3, 30), norm=LogNorm(), cmap='viridis')
axes[0, 1].plot(3, 0.5, 'r*', markersize=15, label='Global minimum')
axes[0, 1].set_title('Beale Function\n(Flat regions)', fontsize=14)
axes[0, 1].set_xlabel('x')
axes[0, 1].set_ylabel('y')
axes[0, 1].legend()

# Saddle
x = np.linspace(-2, 2, 200)
y = np.linspace(-2, 2, 200)
X, Y = np.meshgrid(x, y)
Z = saddle(X, Y)
axes[1, 0].contour(X, Y, Z, levels=30, cmap='RdBu')
axes[1, 0].plot(0, 0, 'r*', markersize=15, label='Saddle point')
axes[1, 0].set_title('Saddle Point Function\n(x^2 - y^2)', fontsize=14)
axes[1, 0].set_xlabel('x')
axes[1, 0].set_ylabel('y')
axes[1, 0].legend()

# Quadratic Bowl
x = np.linspace(-2, 2, 200)
y = np.linspace(-2, 2, 200)
X, Y = np.meshgrid(x, y)
Z = quadratic_bowl(X, Y)
axes[1, 1].contour(X, Y, Z, levels=30, cmap='viridis')
axes[1, 1].plot(0, 0, 'r*', markersize=15, label='Global minimum')
axes[1, 1].set_title('Ill-conditioned Quadratic\n(Condition number = 10)', fontsize=14)
axes[1, 1].set_xlabel('x')
axes[1, 1].set_ylabel('y')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 2. Implementing Optimizers from Scratch

Now let's implement each optimizer from scratch, understanding the intuition and mathematics behind each.

### 2.1 Vanilla Gradient Descent (Baseline)

The simplest form of gradient descent:

$$\theta_{t+1} = \theta_t - \eta \nabla L(\theta_t)$$

where $\eta$ is the learning rate and $\nabla L(\theta_t)$ is the gradient.

In [ ]:
class VanillaGD:
    """Vanilla Gradient Descent optimizer.
    
    The simplest optimization algorithm that updates parameters
    in the direction of steepest descent.
    
    Parameters:
    -----------
    learning_rate : float
        Step size for parameter updates
    """
    
    def __init__(self, learning_rate=0.01):
        self.lr = learning_rate
        self.name = f"Vanilla GD (lr={learning_rate})"
    
    def update(self, params, grads):
        """Update parameters using gradient descent.
        
        Parameters:
        -----------
        params : numpy.ndarray
            Current parameter values
        grads : numpy.ndarray
            Gradients at current parameters
            
        Returns:
        --------
        numpy.ndarray
            Updated parameters
        """
        return params - self.lr * grads
    
    def reset(self):
        """Reset optimizer state (no state for vanilla GD)."""
        pass

# Test vanilla GD
optimizer = VanillaGD(learning_rate=0.001)
params = np.array([1.5, 1.5])
grads = rosenbrock_grad(*params)
new_params = optimizer.update(params, grads)
print(f"Initial params: {params}")
print(f"Gradients: {grads}")
print(f"Updated params: {new_params}")

### 2.2 Momentum

**Intuition**: Momentum helps accelerate gradients in the right direction and dampens oscillations. Think of a ball rolling down a hill - it builds up velocity in consistent directions.

**Mathematics**:
$$v_t = \gamma v_{t-1} + \eta \nabla L(\theta_t)$$
$$\theta_{t+1} = \theta_t - v_t$$

where $\gamma$ is the momentum coefficient (typically 0.9).

**Key insight**: The velocity term accumulates gradients from past steps, providing momentum in directions where gradients point the same way consistently.

In [ ]:
class Momentum:
    """Momentum optimizer.
    
    Accelerates gradient descent by accumulating a velocity vector
    in directions of persistent reduction.
    
    Parameters:
    -----------
    learning_rate : float
        Step size for parameter updates
    momentum : float
        Momentum coefficient (typically 0.9)
    """
    
    def __init__(self, learning_rate=0.01, momentum=0.9):
        self.lr = learning_rate
        self.momentum = momentum
        self.velocity = None
        self.name = f"Momentum (lr={learning_rate}, mom={momentum})"
    
    def update(self, params, grads):
        """Update parameters using momentum.
        
        Parameters:
        -----------
        params : numpy.ndarray
            Current parameter values
        grads : numpy.ndarray
            Gradients at current parameters
            
        Returns:
        --------
        numpy.ndarray
            Updated parameters
        """
        # Initialize velocity if first call
        if self.velocity is None:
            self.velocity = np.zeros_like(params)
        
        # Update velocity: v = gamma * v + eta * grad
        self.velocity = self.momentum * self.velocity + self.lr * grads
        
        # Update parameters: theta = theta - v
        return params - self.velocity
    
    def reset(self):
        """Reset optimizer state."""
        self.velocity = None

# Test momentum
optimizer = Momentum(learning_rate=0.001, momentum=0.9)
params = np.array([1.5, 1.5])

print("Momentum builds up over iterations:")
for i in range(5):
    grads = rosenbrock_grad(*params)
    params = optimizer.update(params, grads)
    print(f"  Step {i+1}: params={params}, velocity={optimizer.velocity}")

### 2.3 AdaGrad (Adaptive Gradient)

**Intuition**: AdaGrad adapts the learning rate for each parameter individually. Parameters with large gradients get smaller learning rates, while parameters with small gradients get larger learning rates.

**Mathematics**:
$$G_t = G_{t-1} + (\nabla L(\theta_t))^2$$
$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{G_t + \epsilon}} \nabla L(\theta_t)$$

where $G_t$ is the sum of squared gradients and $\epsilon$ is a small constant for numerical stability.

**Key insight**: Good for sparse data where you want to learn more from infrequent features. The downside is that the learning rate monotonically decreases and can become too small.

In [ ]:
class AdaGrad:
    """AdaGrad optimizer.
    
    Adapts the learning rate for each parameter based on 
    the historical sum of squared gradients.
    
    Parameters:
    -----------
    learning_rate : float
        Initial step size for parameter updates
    epsilon : float
        Small constant for numerical stability
    """
    
    def __init__(self, learning_rate=0.01, epsilon=1e-8):
        self.lr = learning_rate
        self.epsilon = epsilon
        self.G = None  # Sum of squared gradients
        self.name = f"AdaGrad (lr={learning_rate})"
    
    def update(self, params, grads):
        """Update parameters using AdaGrad.
        
        Parameters:
        -----------
        params : numpy.ndarray
            Current parameter values
        grads : numpy.ndarray
            Gradients at current parameters
            
        Returns:
        --------
        numpy.ndarray
            Updated parameters
        """
        # Initialize accumulated gradient if first call
        if self.G is None:
            self.G = np.zeros_like(params)
        
        # Accumulate squared gradients: G = G + grad^2
        self.G += grads ** 2
        
        # Compute adaptive learning rate and update
        # theta = theta - (eta / sqrt(G + eps)) * grad
        return params - (self.lr / (np.sqrt(self.G) + self.epsilon)) * grads
    
    def reset(self):
        """Reset optimizer state."""
        self.G = None

# Test AdaGrad - show how learning rate adapts
optimizer = AdaGrad(learning_rate=0.5)
params = np.array([1.5, 1.5])

print("AdaGrad adapts learning rate per parameter:")
for i in range(5):
    grads = rosenbrock_grad(*params)
    effective_lr = optimizer.lr / (np.sqrt(optimizer.G if optimizer.G is not None else np.zeros(2)) + optimizer.epsilon)
    params = optimizer.update(params, grads)
    new_effective_lr = optimizer.lr / (np.sqrt(optimizer.G) + optimizer.epsilon)
    print(f"  Step {i+1}: effective_lr={new_effective_lr}, G={optimizer.G}")

### 2.4 RMSprop

**Intuition**: RMSprop addresses AdaGrad's diminishing learning rate problem by using an exponential moving average of squared gradients instead of the sum.

**Mathematics**:
$$E[g^2]_t = \beta E[g^2]_{t-1} + (1-\beta)(\nabla L(\theta_t))^2$$
$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{E[g^2]_t + \epsilon}} \nabla L(\theta_t)$$

where $\beta$ is the decay rate (typically 0.9).

**Key insight**: The exponential moving average allows recent gradients to have more influence, preventing the learning rate from decaying too quickly.

In [ ]:
class RMSprop:
    """RMSprop optimizer.
    
    Maintains a moving average of squared gradients to normalize
    the gradient. This addresses AdaGrad's radically diminishing
    learning rates.
    
    Parameters:
    -----------
    learning_rate : float
        Step size for parameter updates
    decay_rate : float
        Decay rate for the moving average (typically 0.9)
    epsilon : float
        Small constant for numerical stability
    """
    
    def __init__(self, learning_rate=0.01, decay_rate=0.9, epsilon=1e-8):
        self.lr = learning_rate
        self.decay_rate = decay_rate
        self.epsilon = epsilon
        self.Eg2 = None  # Exponential moving average of squared gradients
        self.name = f"RMSprop (lr={learning_rate}, decay={decay_rate})"
    
    def update(self, params, grads):
        """Update parameters using RMSprop.
        
        Parameters:
        -----------
        params : numpy.ndarray
            Current parameter values
        grads : numpy.ndarray
            Gradients at current parameters
            
        Returns:
        --------
        numpy.ndarray
            Updated parameters
        """
        # Initialize moving average if first call
        if self.Eg2 is None:
            self.Eg2 = np.zeros_like(params)
        
        # Update moving average: E[g^2] = beta * E[g^2] + (1-beta) * g^2
        self.Eg2 = self.decay_rate * self.Eg2 + (1 - self.decay_rate) * grads ** 2
        
        # Update parameters: theta = theta - (eta / sqrt(E[g^2] + eps)) * g
        return params - (self.lr / (np.sqrt(self.Eg2) + self.epsilon)) * grads
    
    def reset(self):
        """Reset optimizer state."""
        self.Eg2 = None

# Test RMSprop
optimizer = RMSprop(learning_rate=0.01, decay_rate=0.9)
params = np.array([1.5, 1.5])

print("RMSprop uses exponential moving average:")
for i in range(5):
    grads = rosenbrock_grad(*params)
    params = optimizer.update(params, grads)
    effective_lr = optimizer.lr / (np.sqrt(optimizer.Eg2) + optimizer.epsilon)
    print(f"  Step {i+1}: effective_lr={effective_lr}, E[g^2]={optimizer.Eg2}")

### 2.5 Adam (Adaptive Moment Estimation)

**Intuition**: Adam combines the best of Momentum and RMSprop. It maintains both the first moment (mean) and second moment (uncentered variance) of gradients, with bias correction.

**Mathematics**:
$$m_t = \beta_1 m_{t-1} + (1-\beta_1) \nabla L(\theta_t)$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) (\nabla L(\theta_t))^2$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$$
$$\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

**Key insight**: The bias correction is crucial - without it, the initial moments would be biased towards zero since they're initialized to zero.

In [ ]:
class Adam:
    """Adam optimizer.
    
    Combines momentum with adaptive learning rates. Maintains 
    exponential moving averages of both gradients (first moment)
    and squared gradients (second moment) with bias correction.
    
    Parameters:
    -----------
    learning_rate : float
        Step size for parameter updates
    beta1 : float
        Exponential decay rate for first moment (typically 0.9)
    beta2 : float
        Exponential decay rate for second moment (typically 0.999)
    epsilon : float
        Small constant for numerical stability
    """
    
    def __init__(self, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.lr = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = None  # First moment (mean of gradients)
        self.v = None  # Second moment (uncentered variance)
        self.t = 0     # Timestep
        self.name = f"Adam (lr={learning_rate}, b1={beta1}, b2={beta2})"
    
    def update(self, params, grads):
        """Update parameters using Adam.
        
        Parameters:
        -----------
        params : numpy.ndarray
            Current parameter values
        grads : numpy.ndarray
            Gradients at current parameters
            
        Returns:
        --------
        numpy.ndarray
            Updated parameters
        """
        # Initialize moments if first call
        if self.m is None:
            self.m = np.zeros_like(params)
            self.v = np.zeros_like(params)
        
        # Increment timestep
        self.t += 1
        
        # Update biased first moment: m = beta1 * m + (1 - beta1) * g
        self.m = self.beta1 * self.m + (1 - self.beta1) * grads
        
        # Update biased second moment: v = beta2 * v + (1 - beta2) * g^2
        self.v = self.beta2 * self.v + (1 - self.beta2) * grads ** 2
        
        # Compute bias-corrected moments
        m_hat = self.m / (1 - self.beta1 ** self.t)
        v_hat = self.v / (1 - self.beta2 ** self.t)
        
        # Update parameters: theta = theta - lr * m_hat / (sqrt(v_hat) + eps)
        return params - self.lr * m_hat / (np.sqrt(v_hat) + self.epsilon)
    
    def reset(self):
        """Reset optimizer state."""
        self.m = None
        self.v = None
        self.t = 0

# Test Adam
optimizer = Adam(learning_rate=0.01)
params = np.array([1.5, 1.5])

print("Adam combines momentum with adaptive learning rates:")
for i in range(5):
    grads = rosenbrock_grad(*params)
    params = optimizer.update(params, grads)
    m_hat = optimizer.m / (1 - optimizer.beta1 ** optimizer.t)
    v_hat = optimizer.v / (1 - optimizer.beta2 ** optimizer.t)
    print(f"  Step {i+1}: m_hat={m_hat}, v_hat={v_hat}")

## 3. Optimizer Race Animation

Let's visualize all optimizers racing on the same loss surface. This provides great intuition about their different behaviors.

In [ ]:
def run_optimization(optimizer, func, grad_func, start, n_steps=100):
    """Run optimization and record trajectory.
    
    Parameters:
    -----------
    optimizer : optimizer object
        Optimizer with update method
    func : callable
        Objective function
    grad_func : callable
        Gradient function
    start : numpy.ndarray
        Starting point
    n_steps : int
        Number of optimization steps
        
    Returns:
    --------
    numpy.ndarray
        Array of shape (n_steps+1, 2) containing the trajectory
    list
        List of function values at each step
    """
    optimizer.reset()
    params = start.copy()
    trajectory = [params.copy()]
    values = [func(*params)]
    
    for _ in range(n_steps):
        grads = grad_func(*params)
        params = optimizer.update(params, grads)
        trajectory.append(params.copy())
        values.append(func(*params))
    
    return np.array(trajectory), values

# Test run
optimizer = Adam(learning_rate=0.01)
trajectory, values = run_optimization(
    optimizer, rosenbrock, rosenbrock_grad, 
    np.array([-1.0, -1.0]), n_steps=50
)
print(f"Trajectory shape: {trajectory.shape}")
print(f"Final position: {trajectory[-1]}")
print(f"Final value: {values[-1]:.6f}")

In [ ]:
def create_optimizer_race_animation(func, grad_func, optimizers, start, 
                                     n_steps=200, xlim=(-2, 2), ylim=(-1, 3),
                                     title="Optimizer Race", minimum=(1, 1)):
    """Create an animated comparison of multiple optimizers.
    
    Parameters:
    -----------
    func : callable
        Objective function
    grad_func : callable
        Gradient function
    optimizers : list
        List of optimizer objects
    start : numpy.ndarray
        Starting point for all optimizers
    n_steps : int
        Number of optimization steps
    xlim, ylim : tuple
        Plot limits
    title : str
        Plot title
    minimum : tuple
        Location of the global minimum
        
    Returns:
    --------
    matplotlib.animation.FuncAnimation
        Animation object
    """
    # Run all optimizers and collect trajectories
    trajectories = []
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12']
    
    for opt in optimizers:
        traj, _ = run_optimization(opt, func, grad_func, start, n_steps)
        trajectories.append(traj)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Create contour plot
    x = np.linspace(xlim[0], xlim[1], 200)
    y = np.linspace(ylim[0], ylim[1], 200)
    X, Y = np.meshgrid(x, y)
    Z = func(X, Y)
    
    # Use log scale for better visualization
    levels = np.logspace(-1, 3, 40)
    ax.contour(X, Y, Z, levels=levels, norm=LogNorm(), cmap='viridis', alpha=0.7)
    ax.contourf(X, Y, Z, levels=levels, norm=LogNorm(), cmap='viridis', alpha=0.3)
    
    # Mark start and minimum
    ax.plot(*start, 'ko', markersize=15, label='Start', zorder=10)
    ax.plot(*minimum, 'r*', markersize=20, label='Minimum', zorder=10)
    
    # Initialize lines and points for each optimizer
    lines = []
    points = []
    for i, (opt, traj) in enumerate(zip(optimizers, trajectories)):
        line, = ax.plot([], [], '-', color=colors[i % len(colors)], 
                        linewidth=2, label=opt.name, alpha=0.8)
        point, = ax.plot([], [], 'o', color=colors[i % len(colors)], 
                         markersize=10, zorder=5)
        lines.append(line)
        points.append(point)
    
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_xlabel('x', fontsize=14)
    ax.set_ylabel('y', fontsize=14)
    ax.set_title(title, fontsize=16)
    ax.legend(loc='upper left', fontsize=10)
    
    # Step counter text
    step_text = ax.text(0.02, 0.02, '', transform=ax.transAxes, fontsize=12)
    
    def init():
        for line, point in zip(lines, points):
            line.set_data([], [])
            point.set_data([], [])
        step_text.set_text('')
        return lines + points + [step_text]
    
    def animate(frame):
        for line, point, traj in zip(lines, points, trajectories):
            line.set_data(traj[:frame+1, 0], traj[:frame+1, 1])
            point.set_data([traj[frame, 0]], [traj[frame, 1]])
        step_text.set_text(f'Step: {frame}')
        return lines + points + [step_text]
    
    anim = animation.FuncAnimation(fig, animate, init_func=init,
                                   frames=n_steps+1, interval=50, blit=True)
    plt.close(fig)
    return anim

print("Animation function defined!")

In [ ]:
# Create optimizers with tuned hyperparameters for the Rosenbrock function
optimizers = [
    Momentum(learning_rate=0.001, momentum=0.9),
    AdaGrad(learning_rate=0.5),
    RMSprop(learning_rate=0.01, decay_rate=0.9),
    Adam(learning_rate=0.05, beta1=0.9, beta2=0.999)
]

# Starting point
start = np.array([-1.0, -1.0])

# Create animation
anim = create_optimizer_race_animation(
    rosenbrock, rosenbrock_grad, optimizers, start,
    n_steps=200, xlim=(-2, 2), ylim=(-1.5, 3),
    title="Optimizer Race on Rosenbrock Function",
    minimum=(1, 1)
)

# Display animation
HTML(anim.to_jshtml())

In [ ]:
# Let's also create a static comparison showing the full trajectories

def plot_optimizer_comparison(func, grad_func, optimizers, start,
                               n_steps=200, xlim=(-2, 2), ylim=(-1, 3),
                               title="Optimizer Comparison", minimum=(1, 1)):
    """Create a static plot comparing optimizer trajectories."""
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
    
    # Run optimizers
    trajectories = []
    all_values = []
    for opt in optimizers:
        traj, vals = run_optimization(opt, func, grad_func, start, n_steps)
        trajectories.append(traj)
        all_values.append(vals)
    
    # Left plot: Trajectories on contour
    ax = axes[0]
    x = np.linspace(xlim[0], xlim[1], 200)
    y = np.linspace(ylim[0], ylim[1], 200)
    X, Y = np.meshgrid(x, y)
    Z = func(X, Y)
    
    levels = np.logspace(-1, 3, 40)
    ax.contour(X, Y, Z, levels=levels, norm=LogNorm(), cmap='viridis', alpha=0.7)
    ax.contourf(X, Y, Z, levels=levels, norm=LogNorm(), cmap='viridis', alpha=0.2)
    
    for i, (opt, traj) in enumerate(zip(optimizers, trajectories)):
        ax.plot(traj[:, 0], traj[:, 1], '-', color=colors[i], 
                linewidth=2, label=opt.name, alpha=0.8)
        ax.plot(traj[-1, 0], traj[-1, 1], 'o', color=colors[i], markersize=10)
    
    ax.plot(*start, 'ko', markersize=12, label='Start', zorder=10)
    ax.plot(*minimum, 'r*', markersize=20, label='Minimum', zorder=10)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_xlabel('x', fontsize=14)
    ax.set_ylabel('y', fontsize=14)
    ax.set_title(f'{title} - Trajectories', fontsize=14)
    ax.legend(loc='upper left', fontsize=9)
    
    # Right plot: Loss curves
    ax = axes[1]
    for i, (opt, vals) in enumerate(zip(optimizers, all_values)):
        ax.semilogy(vals, color=colors[i], linewidth=2, label=opt.name)
    
    ax.set_xlabel('Iteration', fontsize=14)
    ax.set_ylabel('Loss (log scale)', fontsize=14)
    ax.set_title(f'{title} - Convergence', fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print final results
    print("\n" + "="*60)
    print("Final Results:")
    print("="*60)
    for opt, traj, vals in zip(optimizers, trajectories, all_values):
        dist = np.linalg.norm(traj[-1] - np.array(minimum))
        print(f"{opt.name:40s} | Final Loss: {vals[-1]:12.6f} | Dist to min: {dist:.6f}")

# Run comparison on Rosenbrock
optimizers = [
    Momentum(learning_rate=0.001, momentum=0.9),
    AdaGrad(learning_rate=0.5),
    RMSprop(learning_rate=0.01, decay_rate=0.9),
    Adam(learning_rate=0.05, beta1=0.9, beta2=0.999)
]

plot_optimizer_comparison(
    rosenbrock, rosenbrock_grad, optimizers, 
    np.array([-1.0, -1.0]),
    n_steps=300, xlim=(-2, 2), ylim=(-1.5, 3),
    title="Rosenbrock Function",
    minimum=(1, 1)
)

In [ ]:
# Try on the ill-conditioned quadratic
optimizers = [
    Momentum(learning_rate=0.01, momentum=0.9),
    AdaGrad(learning_rate=0.5),
    RMSprop(learning_rate=0.1, decay_rate=0.9),
    Adam(learning_rate=0.1, beta1=0.9, beta2=0.999)
]

plot_optimizer_comparison(
    quadratic_bowl, quadratic_bowl_grad, optimizers,
    np.array([1.5, 1.5]),
    n_steps=100, xlim=(-2, 2), ylim=(-2, 2),
    title="Ill-Conditioned Quadratic",
    minimum=(0, 0)
)

## 4. Side-by-Side Comparison Dashboard

Let's create a comprehensive dashboard comparing all optimizers across multiple test functions and metrics.

In [ ]:
def create_optimizer_dashboard(n_steps=200):
    """Create a comprehensive dashboard comparing optimizers."""
    
    # Test functions with their configurations
    test_cases = [
        {
            'name': 'Rosenbrock',
            'func': rosenbrock,
            'grad': rosenbrock_grad,
            'start': np.array([-1.0, -1.0]),
            'minimum': (1, 1),
            'xlim': (-2, 2),
            'ylim': (-1.5, 3)
        },
        {
            'name': 'Quadratic Bowl',
            'func': quadratic_bowl,
            'grad': quadratic_bowl_grad,
            'start': np.array([1.5, 1.5]),
            'minimum': (0, 0),
            'xlim': (-2, 2),
            'ylim': (-2, 2)
        },
        {
            'name': 'Beale',
            'func': beale,
            'grad': beale_grad,
            'start': np.array([0.0, 0.0]),
            'minimum': (3, 0.5),
            'xlim': (-1, 4.5),
            'ylim': (-2, 2)
        }
    ]
    
    # Optimizers with tuned hyperparameters for each test case
    optimizer_configs = [
        {'class': Momentum, 'params': {'learning_rate': 0.001, 'momentum': 0.9}},
        {'class': AdaGrad, 'params': {'learning_rate': 0.5}},
        {'class': RMSprop, 'params': {'learning_rate': 0.01, 'decay_rate': 0.9}},
        {'class': Adam, 'params': {'learning_rate': 0.05}}
    ]
    
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
    
    fig, axes = plt.subplots(len(test_cases), 2, figsize=(16, 5*len(test_cases)))
    
    results = {}
    
    for row, case in enumerate(test_cases):
        results[case['name']] = {}
        
        # Create optimizers fresh for each test case
        optimizers = [cfg['class'](**cfg['params']) for cfg in optimizer_configs]
        
        # Run optimizations
        trajectories = []
        all_values = []
        for opt in optimizers:
            traj, vals = run_optimization(
                opt, case['func'], case['grad'], 
                case['start'], n_steps
            )
            trajectories.append(traj)
            all_values.append(vals)
            results[case['name']][opt.name] = {
                'final_loss': vals[-1],
                'final_pos': traj[-1],
                'dist_to_min': np.linalg.norm(traj[-1] - np.array(case['minimum']))
            }
        
        # Left plot: Trajectories
        ax = axes[row, 0]
        x = np.linspace(case['xlim'][0], case['xlim'][1], 200)
        y = np.linspace(case['ylim'][0], case['ylim'][1], 200)
        X, Y = np.meshgrid(x, y)
        Z = case['func'](X, Y)
        Z = np.clip(Z, 1e-2, 1e4)
        
        ax.contour(X, Y, Z, levels=np.logspace(-1, 3, 30), 
                   norm=LogNorm(), cmap='viridis', alpha=0.7)
        
        for i, (opt, traj) in enumerate(zip(optimizers, trajectories)):
            ax.plot(traj[:, 0], traj[:, 1], '-', color=colors[i], 
                    linewidth=2, label=opt.name.split('(')[0], alpha=0.8)
        
        ax.plot(*case['start'], 'ko', markersize=10)
        ax.plot(*case['minimum'], 'r*', markersize=15)
        ax.set_xlim(case['xlim'])
        ax.set_ylim(case['ylim'])
        ax.set_title(f"{case['name']} - Trajectories", fontsize=14)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        if row == 0:
            ax.legend(loc='upper left', fontsize=9)
        
        # Right plot: Convergence
        ax = axes[row, 1]
        for i, (opt, vals) in enumerate(zip(optimizers, all_values)):
            ax.semilogy(vals, color=colors[i], linewidth=2, 
                        label=opt.name.split('(')[0])
        
        ax.set_title(f"{case['name']} - Convergence", fontsize=14)
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Loss (log scale)')
        ax.grid(True, alpha=0.3)
        if row == 0:
            ax.legend(fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    return results

# Create dashboard
results = create_optimizer_dashboard(n_steps=300)

In [ ]:
# Print summary table
print("\n" + "="*80)
print("OPTIMIZER COMPARISON SUMMARY")
print("="*80)

for func_name, func_results in results.items():
    print(f"\n{func_name}:")
    print("-"*70)
    print(f"{'Optimizer':<35} | {'Final Loss':>15} | {'Dist to Min':>12}")
    print("-"*70)
    for opt_name, metrics in func_results.items():
        short_name = opt_name.split('(')[0].strip()
        print(f"{short_name:<35} | {metrics['final_loss']:>15.6f} | {metrics['dist_to_min']:>12.6f}")

## 5. When to Use Each Optimizer (Decision Guide)

Based on our experiments and theoretical understanding, here's a practical guide for choosing optimizers:

In [ ]:
# Create a visual decision guide

decision_guide = """
+------------------+-------------------+-------------------+-------------------+-------------------+
|    Criterion     |     Momentum      |     AdaGrad       |     RMSprop       |       Adam        |
+==================+===================+===================+===================+===================+
| Sparse Data      |        -          |       +++         |        ++         |        ++         |
+------------------+-------------------+-------------------+-------------------+-------------------+
| Non-stationary   |        +          |        -          |        ++         |       +++         |
+------------------+-------------------+-------------------+-------------------+-------------------+
| Noisy Gradients  |        +          |        +          |        ++         |       +++         |
+------------------+-------------------+-------------------+-------------------+-------------------+
| Ravines/Valleys  |       ++          |        +          |        ++         |       +++         |
+------------------+-------------------+-------------------+-------------------+-------------------+
| Memory Usage     |       Low         |      Medium       |       Low         |      Medium       |
+------------------+-------------------+-------------------+-------------------+-------------------+
| Hyperparameters  |        2          |        1          |        2          |        3          |
+------------------+-------------------+-------------------+-------------------+-------------------+
| Default Choice   |        -          |        -          |        +          |       +++         |
+------------------+-------------------+-------------------+-------------------+-------------------+

Legend: +++ = Excellent, ++ = Good, + = Okay, - = Not recommended
"""

print(decision_guide)

In [ ]:
# Visual flowchart for optimizer selection

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Flowchart boxes
def draw_box(ax, x, y, width, height, text, color='lightblue'):
    rect = plt.Rectangle((x - width/2, y - height/2), width, height, 
                          facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=10, 
            fontweight='bold', wrap=True)

def draw_diamond(ax, x, y, size, text):
    diamond = plt.Polygon([[x, y+size], [x+size, y], [x, y-size], [x-size, y]], 
                          facecolor='lightyellow', edgecolor='black', linewidth=2)
    ax.add_patch(diamond)
    ax.text(x, y, text, ha='center', va='center', fontsize=9, wrap=True)

def draw_arrow(ax, x1, y1, x2, y2, text=''):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    if text:
        mid_x, mid_y = (x1+x2)/2, (y1+y2)/2
        ax.text(mid_x + 0.3, mid_y, text, fontsize=9)

# Title
ax.text(5, 9.5, 'Optimizer Selection Guide', ha='center', fontsize=16, fontweight='bold')

# Start
draw_box(ax, 5, 8.5, 2, 0.5, 'Start', 'lightgreen')

# First decision
draw_diamond(ax, 5, 7, 0.8, 'Need quick\nresults?')
draw_arrow(ax, 5, 8.25, 5, 7.8)

# Adam path (Yes)
draw_box(ax, 7.5, 7, 2, 0.5, 'Use Adam', 'lightcoral')
draw_arrow(ax, 5.8, 7, 6.5, 7, 'Yes')

# Continue path (No)
draw_diamond(ax, 5, 5.5, 0.8, 'Sparse\nfeatures?')
draw_arrow(ax, 5, 6.2, 5, 6.3, 'No')

# AdaGrad path
draw_box(ax, 7.5, 5.5, 2, 0.5, 'Use AdaGrad', 'lightcoral')
draw_arrow(ax, 5.8, 5.5, 6.5, 5.5, 'Yes')

# Continue
draw_diamond(ax, 5, 4, 0.8, 'RNN/LSTM\ntraining?')
draw_arrow(ax, 5, 4.7, 5, 4.8, 'No')

# RMSprop path
draw_box(ax, 7.5, 4, 2, 0.5, 'Use RMSprop', 'lightcoral')
draw_arrow(ax, 5.8, 4, 6.5, 4, 'Yes')

# Continue
draw_diamond(ax, 5, 2.5, 0.8, 'Simple\nconvex?')
draw_arrow(ax, 5, 3.2, 5, 3.3, 'No')

# Momentum path
draw_box(ax, 7.5, 2.5, 2, 0.5, 'Use Momentum', 'lightcoral')
draw_arrow(ax, 5.8, 2.5, 6.5, 2.5, 'Yes')

# Default
draw_box(ax, 5, 1, 2.5, 0.5, 'Default: Adam', 'lightgreen')
draw_arrow(ax, 5, 1.7, 5, 1.5, 'No')

# Notes
notes = """
Key Points:
- Adam is the safe default for most deep learning tasks
- AdaGrad excels with sparse features (NLP, recommender systems)
- RMSprop works well for RNNs and non-stationary objectives
- Momentum is good for simple convex problems with SGD
"""
ax.text(1.5, 2, notes, fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.title('')
plt.tight_layout()
plt.show()

## 6. Hyperparameter Sensitivity Analysis

Let's explore how sensitive each optimizer is to its hyperparameters.

In [ ]:
def hyperparameter_sensitivity_analysis(func, grad_func, start, minimum, 
                                        n_steps=200, n_runs=10):
    """Analyze sensitivity of optimizers to hyperparameter choices."""
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # 1. Momentum: Learning rate sensitivity
    ax = axes[0, 0]
    learning_rates = [0.0001, 0.0005, 0.001, 0.005, 0.01]
    for lr in learning_rates:
        opt = Momentum(learning_rate=lr, momentum=0.9)
        _, vals = run_optimization(opt, func, grad_func, start, n_steps)
        ax.semilogy(vals, label=f'lr={lr}')
    ax.set_title('Momentum: Learning Rate Sensitivity', fontsize=12)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # 2. Momentum: Momentum coefficient sensitivity
    ax = axes[0, 1]
    momentums = [0.0, 0.5, 0.8, 0.9, 0.95, 0.99]
    for mom in momentums:
        opt = Momentum(learning_rate=0.001, momentum=mom)
        _, vals = run_optimization(opt, func, grad_func, start, n_steps)
        ax.semilogy(vals, label=f'mom={mom}')
    ax.set_title('Momentum: Momentum Coefficient Sensitivity', fontsize=12)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # 3. Adam: Learning rate sensitivity
    ax = axes[1, 0]
    learning_rates = [0.0001, 0.001, 0.01, 0.05, 0.1, 0.5]
    for lr in learning_rates:
        opt = Adam(learning_rate=lr)
        _, vals = run_optimization(opt, func, grad_func, start, n_steps)
        ax.semilogy(vals, label=f'lr={lr}')
    ax.set_title('Adam: Learning Rate Sensitivity', fontsize=12)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # 4. Adam: Beta parameters sensitivity
    ax = axes[1, 1]
    beta_configs = [
        (0.5, 0.999, 'b1=0.5, b2=0.999'),
        (0.9, 0.999, 'b1=0.9, b2=0.999 (default)'),
        (0.95, 0.999, 'b1=0.95, b2=0.999'),
        (0.9, 0.99, 'b1=0.9, b2=0.99'),
        (0.9, 0.9999, 'b1=0.9, b2=0.9999'),
    ]
    for b1, b2, label in beta_configs:
        opt = Adam(learning_rate=0.05, beta1=b1, beta2=b2)
        _, vals = run_optimization(opt, func, grad_func, start, n_steps)
        ax.semilogy(vals, label=label)
    ax.set_title('Adam: Beta Parameters Sensitivity', fontsize=12)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Run sensitivity analysis on Rosenbrock
hyperparameter_sensitivity_analysis(
    rosenbrock, rosenbrock_grad,
    np.array([-1.0, -1.0]), (1, 1),
    n_steps=300
)

In [ ]:
# Create heatmap of final loss for different hyperparameter combinations

def create_hyperparameter_heatmap(func, grad_func, start, n_steps=200):
    """Create heatmaps showing optimizer performance across hyperparameter space."""
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Adam: Learning rate vs Beta1
    ax = axes[0]
    lrs = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2]
    beta1s = [0.5, 0.7, 0.8, 0.9, 0.95, 0.99]
    
    results = np.zeros((len(beta1s), len(lrs)))
    for i, b1 in enumerate(beta1s):
        for j, lr in enumerate(lrs):
            opt = Adam(learning_rate=lr, beta1=b1)
            _, vals = run_optimization(opt, func, grad_func, start, n_steps)
            results[i, j] = np.log10(vals[-1] + 1e-10)
    
    im = ax.imshow(results, cmap='RdYlGn_r', aspect='auto')
    ax.set_xticks(range(len(lrs)))
    ax.set_xticklabels([str(lr) for lr in lrs])
    ax.set_yticks(range(len(beta1s)))
    ax.set_yticklabels([str(b) for b in beta1s])
    ax.set_xlabel('Learning Rate', fontsize=12)
    ax.set_ylabel('Beta1', fontsize=12)
    ax.set_title('Adam: Final Loss (log10)', fontsize=14)
    plt.colorbar(im, ax=ax)
    
    # Add annotations
    for i in range(len(beta1s)):
        for j in range(len(lrs)):
            ax.text(j, i, f'{results[i,j]:.1f}', ha='center', va='center', 
                   fontsize=8, color='white' if results[i,j] > 0 else 'black')
    
    # RMSprop: Learning rate vs Decay rate
    ax = axes[1]
    lrs = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2]
    decays = [0.5, 0.7, 0.8, 0.9, 0.95, 0.99]
    
    results = np.zeros((len(decays), len(lrs)))
    for i, decay in enumerate(decays):
        for j, lr in enumerate(lrs):
            opt = RMSprop(learning_rate=lr, decay_rate=decay)
            _, vals = run_optimization(opt, func, grad_func, start, n_steps)
            results[i, j] = np.log10(vals[-1] + 1e-10)
    
    im = ax.imshow(results, cmap='RdYlGn_r', aspect='auto')
    ax.set_xticks(range(len(lrs)))
    ax.set_xticklabels([str(lr) for lr in lrs])
    ax.set_yticks(range(len(decays)))
    ax.set_yticklabels([str(d) for d in decays])
    ax.set_xlabel('Learning Rate', fontsize=12)
    ax.set_ylabel('Decay Rate', fontsize=12)
    ax.set_title('RMSprop: Final Loss (log10)', fontsize=14)
    plt.colorbar(im, ax=ax)
    
    # Add annotations
    for i in range(len(decays)):
        for j in range(len(lrs)):
            ax.text(j, i, f'{results[i,j]:.1f}', ha='center', va='center',
                   fontsize=8, color='white' if results[i,j] > 0 else 'black')
    
    plt.tight_layout()
    plt.show()

create_hyperparameter_heatmap(
    rosenbrock, rosenbrock_grad,
    np.array([-1.0, -1.0]),
    n_steps=300
)

## 7. Practical Tips and Best Practices

### Learning Rate Guidelines

| Optimizer | Typical Starting LR | Range to Try |
|-----------|--------------------|--------------|
| Momentum  | 0.01              | 0.001 - 0.1  |
| AdaGrad   | 0.01              | 0.001 - 0.1  |
| RMSprop   | 0.001             | 0.0001 - 0.01|
| Adam      | 0.001             | 0.0001 - 0.01|

### Common Pitfalls

1. **Learning rate too high**: Loss oscillates or diverges
2. **Learning rate too low**: Training takes forever
3. **Wrong optimizer for the task**: e.g., using AdaGrad for non-convex problems
4. **Not using learning rate schedules**: Often necessary for fine-tuning

In [ ]:
# Demonstrate learning rate schedules

def step_decay(initial_lr, epoch, drop_rate=0.5, epochs_drop=50):
    """Step decay: reduce LR by drop_rate every epochs_drop epochs."""
    return initial_lr * (drop_rate ** (epoch // epochs_drop))

def exponential_decay(initial_lr, epoch, decay_rate=0.95):
    """Exponential decay: LR = initial_lr * decay_rate^epoch."""
    return initial_lr * (decay_rate ** epoch)

def cosine_annealing(initial_lr, epoch, total_epochs):
    """Cosine annealing: smooth decay following cosine curve."""
    return initial_lr * (1 + np.cos(np.pi * epoch / total_epochs)) / 2

def warmup_then_decay(initial_lr, epoch, warmup_epochs=10, total_epochs=100):
    """Linear warmup followed by cosine decay."""
    if epoch < warmup_epochs:
        return initial_lr * (epoch + 1) / warmup_epochs
    else:
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return initial_lr * (1 + np.cos(np.pi * progress)) / 2

# Visualize schedules
fig, ax = plt.subplots(figsize=(12, 6))

epochs = np.arange(100)
initial_lr = 0.1

schedules = [
    ('Step Decay', [step_decay(initial_lr, e) for e in epochs]),
    ('Exponential Decay', [exponential_decay(initial_lr, e) for e in epochs]),
    ('Cosine Annealing', [cosine_annealing(initial_lr, e, 100) for e in epochs]),
    ('Warmup + Cosine', [warmup_then_decay(initial_lr, e) for e in epochs]),
]

for name, lrs in schedules:
    ax.plot(epochs, lrs, linewidth=2, label=name)

ax.set_xlabel('Epoch', fontsize=14)
ax.set_ylabel('Learning Rate', fontsize=14)
ax.set_title('Common Learning Rate Schedules', fontsize=16)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Interview Questions

### Question 1: Explain the key difference between AdaGrad and RMSprop

**Answer:**
The key difference lies in how they accumulate past gradient information:

- **AdaGrad** accumulates the **sum** of all past squared gradients: $G_t = G_{t-1} + g_t^2$
- **RMSprop** uses an **exponential moving average** of squared gradients: $E[g^2]_t = \beta E[g^2]_{t-1} + (1-\beta) g_t^2$

This difference has important practical implications:
- AdaGrad's learning rate monotonically decreases and can become too small, causing training to stall
- RMSprop "forgets" old gradients, allowing the learning rate to increase if recent gradients are small
- RMSprop is better suited for non-stationary objectives and deep networks

### Question 2: Why does Adam need bias correction?

**Answer:**
Adam initializes its moment estimates $m$ and $v$ to zero. This causes the estimates to be biased towards zero, especially during the initial training steps when few gradients have been accumulated.

Without bias correction:
- $m_t \approx (1-\beta_1)g_t$ in early steps (much smaller than true mean)
- $v_t \approx (1-\beta_2)g_t^2$ in early steps

The bias correction terms $(1 - \beta_1^t)$ and $(1 - \beta_2^t)$ counteract this:
- When $t$ is small, these corrections are significant
- As $t \rightarrow \infty$, the corrections approach 1 (no effect)

### Question 3: In what scenarios would you prefer Momentum over Adam?

**Answer:**
Momentum might be preferred over Adam in these scenarios:

1. **Simple convex optimization**: When the loss landscape is well-behaved, Momentum with proper tuning can converge faster

2. **Final fine-tuning**: Some practitioners switch from Adam to SGD+Momentum near the end of training for better generalization

3. **Memory constraints**: Momentum requires storing only one buffer (velocity) vs. two for Adam (first and second moments)

4. **When Adam generalizes poorly**: Research has shown Adam can sometimes converge to sharper minima that generalize worse. SGD+Momentum often finds flatter minima

5. **When you want simpler hyperparameter tuning**: Momentum has fewer hyperparameters (just learning rate and momentum coefficient)

---

## Exercises

### Exercise 1: Implement Nesterov Accelerated Gradient (NAG)

Nesterov momentum looks ahead by computing the gradient at the "lookahead" position:

$$v_t = \gamma v_{t-1} + \eta \nabla L(\theta_t - \gamma v_{t-1})$$
$$\theta_{t+1} = \theta_t - v_t$$

Implement NAG and compare it with standard Momentum on the Rosenbrock function.

In [ ]:
# Exercise 1: Implement NAG here
class NesterovMomentum:
    """Nesterov Accelerated Gradient optimizer.
    
    TODO: Implement the NAG algorithm
    """
    
    def __init__(self, learning_rate=0.01, momentum=0.9):
        self.lr = learning_rate
        self.momentum = momentum
        self.velocity = None
        self.name = f"NAG (lr={learning_rate}, mom={momentum})"
    
    def update(self, params, grads, grad_func=None):
        """Update parameters using Nesterov momentum.
        
        Note: For true NAG, you need the gradient function to compute
        gradient at the lookahead position. For simplicity, you can
        use the approximation: v = mom * v - lr * grad; params += mom * v - lr * grad
        """
        # YOUR CODE HERE
        raise NotImplementedError("TODO: Implement this method")
    
    def reset(self):
        self.velocity = None

# Test your implementation
# nag = NesterovMomentum(learning_rate=0.001, momentum=0.9)
# Compare with standard Momentum on Rosenbrock

### Exercise 2: Implement AdamW (Adam with Decoupled Weight Decay)

AdamW decouples weight decay from the gradient-based update:

$$\theta_{t+1} = \theta_t - \eta \left( \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} + \lambda \theta_t \right)$$

where $\lambda$ is the weight decay coefficient.

In [ ]:
# Exercise 2: Implement AdamW here
class AdamW:
    """AdamW optimizer with decoupled weight decay.
    
    TODO: Implement AdamW
    """
    
    def __init__(self, learning_rate=0.001, beta1=0.9, beta2=0.999, 
                 epsilon=1e-8, weight_decay=0.01):
        # YOUR CODE HERE
        raise NotImplementedError("TODO: Implement this method")
    
    def update(self, params, grads):
        # YOUR CODE HERE
        raise NotImplementedError("TODO: Implement this method")
    
    def reset(self):
        raise NotImplementedError("TODO: Implement this method")

### Exercise 3: Optimizer Benchmark

Create a benchmark that tests all optimizers on:
1. A neural network training task (e.g., MNIST classification)
2. Measure: convergence speed, final accuracy, training time
3. Create a visualization comparing results

In [ ]:
# Exercise 3: Create a neural network benchmark
# Hint: You can use a simple 2-layer network and a synthetic dataset

def create_synthetic_data(n_samples=1000, n_features=10, n_classes=2):
    """Create synthetic classification data."""
    np.random.seed(42)
    X = np.random.randn(n_samples, n_features)
    # Create a simple linear decision boundary with some noise
    w_true = np.random.randn(n_features)
    y = (X @ w_true + np.random.randn(n_samples) * 0.5 > 0).astype(int)
    return X, y

# YOUR CODE HERE: Implement a simple neural network and benchmark the optimizers

---

## Summary

In this notebook, we:

1. **Implemented from scratch** four major optimizers: Momentum, AdaGrad, RMSprop, and Adam
2. **Visualized** how different optimizers navigate loss surfaces through animations
3. **Compared** optimizer performance on multiple test functions
4. **Created** a practical decision guide for optimizer selection
5. **Analyzed** hyperparameter sensitivity for each optimizer

### Key Takeaways

- **Adam** is the safe default choice for most deep learning tasks
- **Momentum** accelerates convergence in consistent gradient directions
- **AdaGrad** adapts learning rates per-parameter, great for sparse data
- **RMSprop** fixes AdaGrad's diminishing learning rate problem
- Learning rate is the most important hyperparameter to tune
- Consider using learning rate schedules for better results